# Teton Classifier Notebook

Random Forest Classifier for the Teton Range, based off of Yang et al. (2023)

## Imports and initializations

In [1]:
import os
import ee
import geemap
import pandas as pd


In [2]:
ee.Authenticate()

True

In [ ]:
EE_PROJECT = "teton-classifier"
# Alpine-ish mask threshold for this first pass. Adjust after inspecting the map.
ALPINE_MIN_ELEV_M = 2200

if EE_PROJECT:
    ee.Initialize(project=EE_PROJECT)
else:
    ee.Initialize()
print("Earth Engine initialized.")

Earth Engine initialized.


## Map, draw study boundary or use preset box


In [6]:
Map = geemap.Map(center=[43.75, -110.75], zoom=10)
aoi = ee.Geometry.Rectangle([-111.05, 43.35, -110.45, 44.05])
Map.addLayer(aoi, {"color": "yellow"}, "AOI")
Map.add_tile_layer(
    url="https://basemap.nationalmap.gov/arcgis/rest/services/USGSTopo/MapServer/tile/{z}/{y}/{x}",
    name="USGS Topo",
    attribution="USGS The National Map",
    opacity=1,
)
Map.addLayer
Map

Map(center=[43.75, -110.75], controls=(WidgetControl(options=['position', 'transparent_bg'], position='toprigh…

## Gather data

In [7]:
features = [
    # Sentinel-2 reflectance
    "B2", "B3", "B4",      # blue, green, red
    "B8",                 # near infrared
    "B11", "B12",         # shortwave infrared

    # Spectral indices
    "NDVI",               # vegetation greenness
    "NDWI",               # water / wetness
    "NDSI",               # snow / ice
    "BSI",                # bare soil / rock tendency

    # Terrain
    "elevation",
    "slope",
    "aspect"
]

### Get persistent snow mask:

In [9]:
persistent_snow = ee.Image(
    "projects/teton-classifier/assets/persistent_snow_mask"
).select(0).rename("persistent_snow")

### Get terrain

In [8]:
terrain = ee.Algorithms.Terrain(
    ee.Image("NASA/NASADEM_HGT/001").select("elevation")
).clip(aoi)

elevation = terrain.select("elevation")
slope = terrain.select("slope")
aspect = terrain.select("aspect")

### Get s2, composite with terrain


In [24]:
def mask_s2_clouds(image):
    qa = image.select("QA60")


    # Identifying relevant bits in the QA60 band for cloud and cirrus detection
    cloud_bit = 1 << 10
    cirrus_bit = 1 << 11

    mask = (
        qa.bitwiseAnd(cloud_bit).eq(0)
        .And(qa.bitwiseAnd(cirrus_bit).eq(0))
    )

    return image.updateMask(mask).divide(10000)

def mask_out_persistent_snow(image):
    return image.updateMask(persistent_snow.neq(1))


s2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(aoi)
    .filterDate("2018-01-01", "2024-12-31")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 60))
    .filter(ee.Filter.calendarRange(227, 257, "day_of_year")) #doys are Aug 15 - Sep 13
)

s2_composite = s2.map(mask_s2_clouds).map(mask_out_persistent_snow).median().clip(aoi)

ndsi = s2_composite.normalizedDifference(["B3", "B11"]).rename("NDSI")
ndwi = s2_composite.normalizedDifference(["B3", "B8"]).rename("NDWI")
ndvi = s2_composite.normalizedDifference(["B8", "B4"]).rename("NDVI")
bsi = s2_composite.expression(
    "((SWIR + RED) - (NIR + BLUE)) / ((SWIR + RED) + (NIR + BLUE))",
    {
        "SWIR": s2_composite.select("B11"),
        "RED": s2_composite.select("B4"),
        "NIR": s2_composite.select("B8"),
        "BLUE": s2_composite.select("B2"),
    }
).rename("BSI")

feature_image = s2_composite.select([
    "B2", "B3", "B4", "B8", "B11", "B12"
]).addBands([
    ndvi,
    ndwi,
    ndsi,
    bsi,
    elevation,
    slope,
    aspect
])

feature_image.bandNames().getInfo()


['B2',
 'B3',
 'B4',
 'B8',
 'B11',
 'B12',
 'NDVI',
 'NDWI',
 'NDSI',
 'BSI',
 'elevation',
 'slope',
 'aspect']

### Build stable water mask

This section removes water from the random-forest training problem and handles it as a physically constrained mask. The goal is to prevent shaded steep slopes from being classified as water.

We use JRC Global Surface Water occurrence, then constrain it with slope. Later, the RF trains only on terrestrial classes, and this water mask is burned back into the final baseline as class `0`.


In [56]:
# Stable water mask for baseline classification.
# JRC occurrence is 0-100: how often water was observed historically.
water_occurrence = ee.Image("JRC/GSW1_4/GlobalSurfaceWater").select("occurrence").clip(aoi)

WATER_OCCURRENCE_MIN = 10
WATER_MAX_SLOPE_DEG = 10

s2_water = (
    ndwi.gt(0.2)
    .And(s2_composite.select("B8").lt(0.12))
    .And(slope.lt(10))
    .rename("s2_water")
)

stable_water = (
    water_occurrence.gte(WATER_OCCURRENCE_MIN)
    .And(slope.lt(WATER_MAX_SLOPE_DEG))
    .rename("stable_water")
    .clip(aoi)
)

stable_water = (
    stable_water
    .Or(s2_water)
    .rename("stable_water")
)


# Debug: areas that look water-like in the global water product but are too steep.
# These are likely terrain/shadow artifacts or unsuitable water candidates.
water_rejected_by_slope = (
    water_occurrence.gte(WATER_OCCURRENCE_MIN)
    .And(slope.gte(WATER_MAX_SLOPE_DEG))
    .selfMask()
    .rename("water_rejected_by_slope")
)

# Print a quick area summary.
water_area_km2 = ee.Image.pixelArea().divide(1e6).updateMask(stable_water).reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=aoi,
    scale=30,
    maxPixels=1e13,
    tileScale=4,
).getInfo()

rejected_water_area_km2 = ee.Image.pixelArea().divide(1e6).updateMask(water_rejected_by_slope).reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=aoi,
    scale=30,
    maxPixels=1e13,
    tileScale=4,
).getInfo()

print("Stable water area km2:", water_area_km2)
print("Rejected steep water-candidate area km2:", rejected_water_area_km2)

# Debug map for the water mask.
WaterDebugMap = geemap.Map(center=[43.75, -110.75], zoom=10)

WaterDebugMap.addLayer(aoi, {"color": "yellow"}, "AOI")
WaterDebugMap.addLayer(s2_water.selfMask(), {"palette": ["00a2ff"]}, "S2 water mask", False)
WaterDebugMap.addLayer(water_occurrence, {"min": 0, "max": 100, "palette": ["ffffff", "7ec8ff", "0046b8"]}, "JRC water occurrence", False)
WaterDebugMap.addLayer(slope, {"min": 0, "max": 60, "palette": ["ffffff", "f4a261", "b00020"]}, "Slope", False)
WaterDebugMap.addLayer(stable_water.selfMask(), {"palette": ["00a2ff"]}, "Stable water mask")
WaterDebugMap.addLayer(water_rejected_by_slope, {"palette": ["ff0000"]}, "Rejected: water occurrence on steep slopes", False)

WaterDebugMap.addLayer(ndwi.gt(0.0).selfMask(), {"palette": ["blue"]}, "NDWI > 0.0", False)
WaterDebugMap.addLayer(s2_composite.select("B8").lt(0.18).selfMask(), {"palette": ["purple"]}, "B8 < 0.18", False)
WaterDebugMap.addLayer(slope.lt(10).selfMask(), {"palette": ["green"]}, "Slope < 10", False)

WaterDebugMap.add_tile_layer(
    url="https://basemap.nationalmap.gov/arcgis/rest/services/USGSTopo/MapServer/tile/{z}/{y}/{x}",
    name="USGS Topo",
    attribution="USGS The National Map",
    opacity=0.45,
)
WaterDebugMap.addLayerControl()
WaterDebugMap


Stable water area km2: {'area': 146.05503638800502}
Rejected steep water-candidate area km2: {'area': 3.149273188033758}


Map(center=[43.75, -110.75], controls=(WidgetControl(options=['position', 'transparent_bg'], position='toprigh…

In [70]:
#Debug map

WaterDebugMap = geemap.Map(center=[43.75, -110.75], zoom=10)

WaterDebugMap.add_tile_layer(
    url="https://basemap.nationalmap.gov/arcgis/rest/services/USGSTopo/MapServer/tile/{z}/{y}/{x}",
    name="USGS Topo",
    attribution="USGS The National Map",
    opacity=0.35,
)

#WaterDebugMap.addLayer(aoi, {"color": "yellow"}, "AOI")

WaterDebugMap.addLayer(
    ndwi,
    {"min": -0.5, "max": 0.5, "palette": ["brown", "white", "blue"]},
    "NDWI"
)

WaterDebugMap.addLayer(
    ndwi.gt(0.0).selfMask(),
    {"palette": ["blue"]},
    "NDWI > 0.0"
)

WaterDebugMap.addLayer(
    s2_composite.select("B8").lt(0.18).selfMask(),
    {"palette": ["purple"]},
    "B8 < 0.18"
)

WaterDebugMap.addLayer(
    slope.lt(10).selfMask(),
    {"palette": ["green"]},
    "Slope < 10"
)

WaterDebugMap.addLayer(
    s2_water.selfMask(),
    {"palette": ["00a2ff"]},
    "S2 water mask"
)

WaterDebugMap.addLayer(
    stable_water.selfMask(),
    {"palette": ["cyan"]},
    "Stable water mask"
)
ndwi_slope_mask = (
    ndwi.gt(0.0)
    .And(slope.lt(10))
    .And(persistent_snow.neq(1))
    .And(ndsi.lt(0.9))
    .selfMask()
)

WaterDebugMap.addLayer(
    ndwi_slope_mask,
    {"palette": ["00a2ff"]},
    "NDWI > 0.0 and slope < 10 and not persistent snow and NDSI < 0.9"
)

WaterDebugMap.addLayerControl()
WaterDebugMap

Map(center=[43.75, -110.75], controls=(WidgetControl(options=['position', 'transparent_bg'], position='toprigh…

# Get DW 

In [ ]:
dw = (
    ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")
    .filterBounds(aoi)
    .filterDate("2023-01-01", "2024-12-31") #to save compute
    .filter(ee.Filter.calendarRange(227, 257, "day_of_year"))
)

dw_probability_bands = [
    "water",
    "trees",
    "grass",
    "flooded_vegetation",
    "crops",
    "shrub_and_scrub",
    "built",
    "bare",
    "snow_and_ice",
]

dw_probabilities = (
    dw
    .select(dw_probability_bands)
    .median()
    .clip(aoi)
)

dw_confidence = dw_probabilities.reduce(ee.Reducer.max()).rename("dw_confidence")

confidence_mask = dw_confidence.gte(0.60)

dw_classes = {
    0: "water",
    1: "trees",
    2: "grass",
    3: "flooded_vegetation",
    4: "crops",
    5: "shrub_and_scrub",
    6: "built",
    7: "bare",
    8: "snow_and_ice",
}

#To remove noise: these are the classes we care about, at least for alpine classification: 
# water, trees, grass, shrub/scrub, bare, snow/ice. Ignoring flooded vegetation, crops, and built areas \

keep_classes = [1, 2, 5, 7]

label_image = dw.select("label").mode().clip(aoi).rename("class")
label_image = label_image.updateMask(persistent_snow.neq(1))
label_image = label_image.updateMask(stable_water.Not())
label_image = label_image.updateMask(confidence_mask)
class_mask = label_image.remap(
    keep_classes,
    [1, 1, 1, 1],
    0
).eq(1)

label_image = label_image.updateMask(class_mask)

training_image = feature_image.addBands(label_image)

#sample training points

samples = training_image.stratifiedSample(
    numPoints=200,
    classBand="class",
    region=aoi,
    scale=10,
    seed=42,
    geometries=True,
    tileScale=4
)

# Check the number of samples and their class distribution -- triggers server computation
samples.aggregate_histogram("class").getInfo()

samples = samples.randomColumn("random", seed=7)
training_samples = samples.filter(ee.Filter.lt("random", 0.7))
validation_samples = samples.filter(ee.Filter.gte("random", 0.7))

In [32]:
samples.aggregate_histogram("class").getInfo()

{'0': 200, '1': 200, '2': 200, '5': 200, '7': 200}

### Train RF Model

In [37]:
feature_names = feature_image.bandNames()

rf = ee.Classifier.smileRandomForest(
    numberOfTrees=100,
    seed=42
).train(
    features=training_samples,
    classProperty="class",
    inputProperties=feature_names
)

# Raw RF output contains only terrestrial classes: trees, grass, shrub/scrub, bare.
baseline_raw = feature_image.classify(rf).rename("baseline_raw")

# Burn stable water and persistent snow/ice back into the baseline map.
baseline_classified = (
    baseline_raw
    .where(stable_water, 0)
    .where(persistent_snow.eq(1), 8)
    .rename("baseline_class")
)

# Keep this alias so older plotting/export cells that reference `classified` still work.
classified = baseline_classified

# Debug: if this lights up, the raw model is still trying to call steep terrain water.
# It should be empty now because water is not in RF training classes.
raw_water_on_steep_slopes = baseline_raw.eq(0).And(slope.gt(15)).selfMask().rename("raw_water_on_steep_slopes")


### Plot results

In [39]:
class_palette = [
    "419bdf",  # 0 water: medium blue
    "397d49",  # 1 trees: dark green
    "88b053",  # 2 grass: light olive green
    "7a87c6",  # 3 flooded vegetation: muted blue-purple
    "e49635",  # 4 crops: orange
    "dfc35a",  # 5 shrub/scrub: yellow-tan
    "c4281b",  # 6 built: red
    "a59b8f",  # 7 bare: gray-beige
    "b39fe1",  # 8 snow/ice: light purple
]

# Resets map layers!

Map = geemap.Map(center=[43.75, -110.75], zoom=10)
Map.addLayer(aoi, {"color": "yellow"}, "AOI")
Map.addLayer(s2_composite, {"bands": ["B4", "B3", "B2"], "min": 0.02, "max": 0.3}, "Sentinel-2 true color")
Map.addLayer(terrain.select("elevation"), {"min": 1800, "max": 4200, "palette": ["2166ac", "f7f7f7", "b2182b"]}, "Elevation")
Map.addLayer(label_image, {"min": 0, "max": 8, "palette": class_palette}, "Dynamic World weak labels")
Map.addLayer(baseline_raw, {"min": 1, "max": 7, "palette": class_palette[1:8]}, "Raw terrestrial RF output", False)
Map.addLayer(stable_water.selfMask(), {"palette": ["00a2ff"]}, "Stable water mask", False)
Map.addLayer(persistent_snow.selfMask(), {"palette": ["ffffff"]}, "Persistent snow mask", False)
Map.addLayer(classified, {"min": 0, "max": 8, "palette": class_palette}, "Final baseline: RF + water + snow")
Map.addLayer(raw_water_on_steep_slopes, {"palette": ["red"]}, "Debug: raw water on steep slopes", False)

Map.add_tile_layer(
    url="https://basemap.nationalmap.gov/arcgis/rest/services/USGSTopo/MapServer/tile/{z}/{y}/{x}",
    name="USGS Topo",
    attribution="USGS The National Map",
    opacity=0.35,
)

# Map.addLayer(training_samples, {"color": "white"}, "Training points", False)
# Map.addLayer(validation_samples, {"color": "black"}, "Validation points", False)

Map.addLayerControl()
Map

Map(center=[43.75, -110.75], controls=(WidgetControl(options=['position', 'transparent_bg'], position='toprigh…